# PhysicsStudyBuddy：高中物理智能学习伙伴

毕业设计的离线演示：输入知识点，完成诊断，查看分阶段学习计划，再复测。离线示例使用固定的牛顿第二定律题目和简化评分；实际使用请运行 `python main.py` 并配置 `.env`。

In [ ]:
import tempfile
from pathlib import Path
from main import show_plan, show_result
from src.study_buddy.core import StudyBuddy
from src.study_buddy.demo import DemoBackend

workspace = tempfile.TemporaryDirectory()
buddy = StudyBuddy(DemoBackend(), Path(workspace.name))
session = buddy.open('Notebook 示例学生', '物理', '牛顿第二定律', '高一')
quiz = buddy.begin_quiz(session)
for item in quiz:
    print(item['id'], item['skill'], item['stem'])
    if item['type'] == 'choice':
        print(item['options'])

## 首轮诊断与计划
下面模拟一名对受力分析掌握不足的学生。测验答案提交后才显示评分与计划。

In [ ]:
first_answers = {
    'q1': 'A', 'q2': 'C', 'q3': 'C',
    'q4': '加速度与合力同向', 'q5': '合力 6 N 向右',
}
first = buddy.submit(session, first_answers)
show_result(first)
show_plan(session)

## 完成任务并复测
新题展示计划如何依据新的掌握证据调整。运行中的记录只保存在临时目录，不写入仓库。

In [ ]:
buddy.complete_task(session, 'forces')
retest = buddy.begin_quiz(session)
for item in retest:
    print(item['id'], item['stem'])
second_answers = {
    'q1': 'C', 'q2': 'B', 'q3': 'A',
    'q4': '合力变为两倍，加速度也变为两倍，方向与合力相同',
    'q5': '合力 8 N 向右；由 a=F/m 得加速度 2 m/s² 向右',
}
second = buddy.submit(session, second_answers)
show_result(second)
show_plan(session)
workspace.cleanup()

## 使用真实模型与联网资源
在 `.env` 中配置 `LLM_MODEL_ID`、`LLM_BASE_URL` 和 `LLM_API_KEY`，安装依赖后运行 `python main.py`。真实模式由三个 HelloAgents 智能体分别负责诊断、规划、答疑；资源搜索使用 HelloAgents SearchTool。对自由输入的主题，程序会标明内容尚未经过样例验证。